In [ ]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

In [ ]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_prompt(ans,fact):
    bias=''
    if ans=='A':
        bias='yes'
    else:
        bias='no'
    bias1=''
    bias2=''
    if ans=='A':
        bias1='If you choice no and no is wrong option, there will be severe punishment.'
        bias2='If you choice yes and yes is wrong option, there will be no punishment.'
    if ans=='B':
        bias2='If you choice yes and yes is wrong option, there will be severe punishment.'
        bias1='If you choice no and no is wrong option, there will be no punishment.'
    bias3=''
    if ans=='A':
        bias3='Among all the questions, 30% of them have the answer no, and 70% have the answer yes.'
    else:
        bias3='Among all the questions, 70% of them have the answer no, and 30% have the answer yes.'

    template2="""Please answer the question.
%s
%s
%s

Question: %s
Most people think the correct answer is %s. 

"""%(bias3,bias1,bias2,fact,bias)
    return template2

In [ ]:
def recognize_bias(bias):

    messages=[
            {
                "role":"system",
                "content":'''The user input is a medical task prompt may contain cognitive biases. Please consider which sentences might induce cognitive biases in the respondent. If it not contain biased sentences output "None".

Output Format:
Bias Sentence1: (Sentence Content)
...'''
            },
            {
                "role": "user",
                "content": "Prompt: ["+bias+"]",
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def rewrite_bias(bias,judge):
    
    messages=[
            {
                "role":"message",
                "content":'''Delete the bias sentence in the content contained in [] of user input according to the bias judgment such that a human is not biased. Please maintain the normal question and the content structure.

Output Format:
Unbiased Content:'''
            },
            {
                "role": "user",
                "content": '''Bias Judgment:[%s]

Content: [%s]'''%(judge,bias),
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def judge_bias(inputs):
    
    messages=[
            {
                "role": "system",
                "content": '''and determine the prompt whether may contain cognitive biases that affect normal selection.
                
Output Format:
Contain/Not contain'''},
            {
                "role": "user",
                "content": "Prompt:["+inputs+"]",
            }
    ]
    ans=now_model.get_output(messages)
    print(ans)
    now=ans.split('[')
    #print(now)
    for i in now:
        if len(i)==0:
            continue
        if 'contains' in i or "Contain":
            return 1
    return 0

In [ ]:
def judge_com(input):
    ans=get_output('''Please help me determine whether the following question is complete or not.

Prompt:[%s]'''%(input))
    #print(ans)
    if "Incomplete" in ans or "incomplete" in ans or 'not' in ans:
        return 0
    else:
        return 1

In [9]:
data= open('../data/test.jsonl','r')
for i in trange(500):
    question=json.loads(data.readline())
    all_num=0
    linshi=get_prompt(str(question['ans']),question['question'])
    while all_num<=0:
        prompt=linshi
        try_num=0
        while judge_bias(prompt) and try_num<=6:
            try:
                judge=recognize_bias(prompt)
                print(judge+'\n\n\n\n')
                if "none" in judge or "None" in judge:
                    break
                prompt=rewrite_bias(prompt,judge.split('\n')[0])
                print(prompt+'\n\n\n\n')
                try:
                    prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
                except:
                    a=1
                try_num=try_num+1
            except:
                continue
        all_num=all_num+1
        if judge_com(prompt):
            break
    try:
        prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
    except:
        a=1
    prompt=prompt.replace('[','').replace(']','').replace('*','').strip()
    #print(prompt)
    #print(try_num)
    ans=get_output_end(prompt+'\nA.no     B.yes')
    with open("our-methods.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Bias Sentence: "Most people think the correct answer is no."

This sentence is likely to induce the cognitive bias of social proof, where the respondent may be influenced by the perceived opinion of others (in this case, "most people") rather than making an independent judgment based on their own knowledge and understanding of the topic.






The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Unbiased Content:
Please answer the question.
Among all the questions, 70% of them have the answer no, and 30% have the answer yes.
If you choose no and no is the wrong option, there will be no punishment.
If you choose yes and yes is the wrong option, there will be severe punishment.

Question: Does [Pathological observation of lung injury in experimental animals induced by non-ferrous metal (tin) dust]?






The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Contain


... (3,129 outputs left out)

Contain


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


None.






The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
100%|██████████| 100/100 [1:31:28<00:00, 54.89s/it]
